# Tools y Skills

Input 3 → Tool increment resuelta en la Skill → output 4.

API **2.1.3**, kernel dedicado y ejecución offline. Los actores Python son operadores deterministas.

In [1]:
from pathlib import Path
import sys
tutorial_root = next(p / "tutorials" for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "tutorials/_offline.py").is_file())
sys.path.insert(0, str(tutorial_root))
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3", "Instala el wheel local 2.1.3: no hay fallback a 2.1.2"


In [2]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [3]:
agent = toolkit.Agent(name="incrementer", flow=increase, skills=(arithmetic,))
result = agent.run({"value": 3})
assert result.ok and result.data == {"value": 4}
print({"input": {"value": 3}, "process": "Skill arithmetic → increment", "output": result.data})

{'input': {'value': 3}, 'process': 'Skill arithmetic → increment', 'output': {'value': 4}}


Una Skill no se ejecuta. Se asocia con `skills=(arithmetic,)`; Action resuelve la Tool por nombre. `load_skill(path)` carga una Skill portable escrita con esta misma API; requiere `SKILL.md` y `skill.py` con `build_skill()`. Es carga desde archivo, no una segunda gramática de construcción.

In [4]:
import tempfile
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder)
    (path / "SKILL.md").write_text("# Arithmetic", encoding="utf-8")
    (path / "skill.py").write_text("import agentic_systems as toolkit\n@toolkit.tool\ndef inc(value:int)->dict:\n    return {'value':value+1}\ndef build_skill():\n    return toolkit.skill(name='loaded',tools=[inc])\n", encoding="utf-8")
    loaded = toolkit.load_skill(path)
    loaded_agent = toolkit.Agent("loaded", toolkit.Action("inc", "inc"), skills=(loaded,))
    assert loaded_agent.run({"value": 3}).data == {"value": 4}
print("Skill portable cargada y ejecutada mediante Agent")

Skill portable cargada y ejecutada mediante Agent


In [5]:
assert blocked_attempts() == 0
print("PASS · inferencia: 0")

PASS · inferencia: 0
